# Create a Simple Reflex-Based Lunar Lander Agent

In this example, we will use Gymnasium, an environment to train agents via reinforcement learning (RL). We will not use RL here but just use the environment with a custom simple reflex-based agent.

## Install Gymnasium

The documentation for Gymnasium is available at https://gymnasium.farama.org/

Steps:
1. Create a new folder and open it with VS Code and install all needed Python Extensions in VS Code.
2. Create a new virtual environment (CTRL-Shift P Python Create Environment...)
3. I needed to install swig and the Python C++ headers on WSL2 via the terminal
    * `sudo apt install swig`
    * `sudo apt-get install python3-dev`
4. Install python libraries for gymnasium with the needed extras

In [205]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
%pip install -q "swig>=4,<5" "gymnasium[box2d,classic-control]>=1.0,<2"

Additional installs for screen capturing so environment visualizations can be converted to embedded videos.
For recording videos, I had to install
    
* `sudo apt-get install xvfb ffmpeg`

Additional python packages for screen capturing.

In [206]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
%pip install -q "pyvirtualdisplay>=3,<4" "gymnasium[other]>=1.0,<2"

In [207]:
# Cài đặt công cụ hệ thống Linux (Bắt buộc để màn hình ảo hoạt động)
!apt-get install -y xvfb ffmpeg > /dev/null 2>&1

## The Lunar Lander Environment

![Luna Lander image](https://gymnasium.farama.org/_images/lunar_lander.gif)

The documentation of the environment is available at: https://gymnasium.farama.org/environments/box2d/lunar_lander/

* Performance Measure: A reward of -100 or +100 points for crashing or landing safely respectively. We do not use
  intermediate rewards here.

* Environment: This environment is a classic rocket trajectory optimization problem. A ship needs to land safely. The space is **continuous** with
  x and y coordinates in the range [-2.5, 2.5]. The landing pad is at coordinate (0,0).

* Actuators:  According to Pontryagin’s
  maximum principle, it is optimal to fire the engine at full throttle or turn it off. This is the reason why this environment has discrete actions: engine on or off. There are four discrete actions available:

    - 0: do nothing
    - 1: fire left orientation engine
    - 2: fire main engine
    - 3: fire right orientation engine

* Sensors: Each observation is an 8-dimensional vector: the coordinates of the lander in x & y, its linear velocities in x & y, its angle, its angular velocity, and two booleans that represent whether each leg is in contact with the ground or not.

The different ranges/settings of the environment can be queried.

In [208]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2)

In [209]:
def query_environment(name):
    env = gym.make(name)
    print(f"Action Space: {env.action_space}")
    print(f"Observation Space: {env.observation_space}")
    print(f"Max Episode Steps: {env.spec.max_episode_steps}")
    print(f"Nondeterministic: {env.spec.nondeterministic}")
   # print(f"Reward Range: {env.reward_range}")
    print(f"Reward Threshold: {env.spec.reward_threshold}")
    env.close()

query_environment("LunarLander-v3")

Action Space: Discrete(4)
Observation Space: Box([ -2.5   -2.5  -10.   -10.    -6.28 -10.    -0.    -0.  ], [ 2.5   2.5  10.   10.    6.28 10.    1.    1.  ], (8,), float32)
Max Episode Steps: 1000
Nondeterministic: False
Reward Threshold: 200



Gymnasium environments are implemented as classes with a `make` method to create the environment, a `reset` method, and a `step` method to execute an action.
To use it with an agent function that expects percepts and returns an action, we need write glue code that connects the environment with the agent function.

In [210]:
def run_episode(agent_function, env, max_steps=1000, verbose = True, render = True):
    """Run one episode in the environment using the provided agent."""

    # Reset the environment to generate the first observation (use seed=42 in reset to get reproducible results)
    observation, info = env.reset()

    # run one episode
    for _ in range(max_steps):
        # call the agent function to select an action
        action = agent_function(observation)

        if verbose:
            print (f"Obs: {observation} -> Action: {action}")

        # step: execute an action in the environment
        observation, reward, terminated, truncated, info = env.step(action)

        # render the environment
        if render:
            env.render()

        if terminated:
            if verbose:
                print(f"Final Reward: {reward}")
            break

    return reward

## Example: A Random Agent

We randomly return one of the actions. The environment accepts the integers 0-3.


In [211]:
def random_agent_function(observation):
    """A random agent that selects actions uniformly at random. It ignores the observation."""
    return np.random.choice([0, 1, 2, 3], p=[0.25, 0.25, 0.25, 0.25])

Run an episode.

In [212]:
env = gym.make("LunarLander-v3", render_mode="human")

run_episode(random_agent_function, env)

env.close()

Obs: [ 0.    1.42  0.2   0.29 -0.   -0.04  0.    0.  ] -> Action: 1
Obs: [ 0.    1.42  0.19  0.26 -0.   -0.01  0.    0.  ] -> Action: 2
Obs: [ 0.01  1.43  0.2   0.27 -0.   -0.01  0.    0.  ] -> Action: 2
Obs: [ 0.01  1.44  0.2   0.27 -0.   -0.    0.    0.  ] -> Action: 3
Obs: [ 0.01  1.44  0.21  0.25 -0.01 -0.05  0.    0.  ] -> Action: 2
Obs: [ 0.01  1.45  0.21  0.29 -0.01 -0.05  0.    0.  ] -> Action: 3
Obs: [ 0.01  1.45  0.22  0.26 -0.01 -0.08  0.    0.  ] -> Action: 0
Obs: [ 0.02  1.46  0.22  0.23 -0.02 -0.08  0.    0.  ] -> Action: 0
Obs: [ 0.02  1.46  0.22  0.21 -0.02 -0.08  0.    0.  ] -> Action: 3
Obs: [ 0.02  1.47  0.23  0.18 -0.03 -0.13  0.    0.  ] -> Action: 0
Obs: [ 0.02  1.47  0.23  0.15 -0.03 -0.13  0.    0.  ] -> Action: 3
Obs: [ 0.03  1.47  0.24  0.13 -0.04 -0.17  0.    0.  ] -> Action: 1
Obs: [ 0.03  1.48  0.23  0.1  -0.05 -0.13  0.    0.  ] -> Action: 2
Obs: [ 0.03  1.48  0.22  0.14 -0.06 -0.14  0.    0.  ] -> Action: 3
Obs: [ 0.03  1.48  0.23  0.12 -0.06 -0.18  0.   

Gymnasium displays environments using `render()` method on the local display. Headless installations like Google Colab do not have a display, but the output can be captured using a virtual display as a video and then add the video to the notebook.

We need to start a virtual display.

Now we can use wrappers for the environment to record the display. The functions are provided in
[display_record.py].



In [213]:
# download if missing
import urllib.request
import os

def download(file, base_url):
    if not os.path.exists(file):
        urllib.request.urlretrieve(base_url + file, file)

download("gymnasium_display_recorder.py",
         "https://raw.githubusercontent.com/mhahsler/Introduction_to_Artificial_Intelligence/refs/heads/master/Agents/")

In [214]:
import gymnasium_display_recorder as dr

env = dr.gym_make('LunarLander-v3', 'LL1', render_fps=30)
run_episode(random_agent_function, env, verbose=False)
env.close()

dr.show('LL1')

## A Simple Reflex-Based Agent

To make the code easier to read, we use enumerations for actions (integers) and observations (index in the observation vector).

In [215]:
from enum import Enum

class Act(Enum):
    LEFT = 1
    RIGHT = 3
    MAIN = 2
    NO_OP = 0

class Obs(Enum):
    X = 0
    Y = 1
    VX = 2
    VY = 3
    ANGLE = 4
    ANGULAR_VELOCITY = 5
    LEFT_LEG_CONTACT = 6
    RIGHT_LEG_CONTACT = 7


Define a simple agent that uses the main thruster to reduce the falling speed if it gets too fast.

In [216]:
def rocket_agent_function(observation):
    """A simple agent function."""

    # run the main thruster, if the lander is falling too fast
    if observation[Obs.VY.value] < -.4:
        return Act.MAIN.value

    return Act.NO_OP.value

In [223]:
env = dr.gym_make('LunarLander-v3', 'LL2', render_fps=30)
run_episode(rocket_agent_function, env, verbose = False)
env.close()

dr.show('LL2')

## Evaluating the Agent

Run the agent on 100 problems and report the average reward.

In [218]:
def run_episodes(agent_function, env, n=1000):
    """Run multiple episodes with the given agent and return the rewards for each episode."""
    return [run_episode(agent_function, env, verbose=False, render=False) for _ in range(n)]

Run experiments.

In [219]:
env = gym.make("LunarLander-v3", render_mode=None)

rewards = run_episodes(rocket_agent_function, env)
print(rewards)

print(f"Average reward: {np.average(rewards)}")
print(f"Success rate: {np.sum(np.array(rewards) == 100)}/{len(rewards)}")

[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

This is not great performance!


## Implement A Better Reflex-Based Agent

Build a better that uses its right and left thrusters to land the craft (more) safely. Test your agent function using 100 problems.

In [224]:
# Code goes here

# ==============================================================================
# YÊU CẦU 3: Better Reflex-Based Agent
# Sử dụng kết hợp: Vị trí ngang (X), Vận tốc ngang (VX), Góc nghiêng (ANGLE, ANGULAR_VELOCITY)
# ==============================================================================

def better_agent_function(observation):
    """Reflex agent cải tiến dùng cả động cơ trái/phải và động cơ chính."""
    x = observation[Obs.X.value]
    y = observation[Obs.Y.value]
    vx = observation[Obs.VX.value]
    vy = observation[Obs.VY.value]
    angle = observation[Obs.ANGLE.value]
    w = observation[Obs.ANGULAR_VELOCITY.value]

    # 1. Tính góc nghiêng mục tiêu dựa trên độ lệch ngang và vận tốc trôi ngang
    # Đang ở bên phải (x > 0) -> hơi nghiêng sang phải để lực đẩy đẩy tàu lùi về trái
    target_angle = x * 0.5 + vx * 1.0
    target_angle = np.clip(target_angle, -0.4, 0.4)

    # 2. Luật cân bằng góc và điều hướng ngang (Dùng động cơ trái/phải)
    angle_error = (angle - target_angle) + w * 0.5

    if angle_error > 0.08:
        return Act.RIGHT.value
    elif angle_error < -0.08:
        return Act.LEFT.value

    # 3. Luật hãm tốc độ rơi (Dùng động cơ chính)
    # Càng xuống thấp (y nhỏ), tốc độ rơi cho phép càng nhỏ
    target_vy = -np.clip(y * 0.75, 0.1, 0.5)

    if vy < target_vy and abs(angle) < 0.4:
        return Act.MAIN.value

    return Act.NO_OP.value

In [225]:
# ==============================================================================
# Ghi hình và hiển thị video 1 lượt chơi (dùng lại dr.gym_make và dr.show có sẵn)
# ==============================================================================
env_record = dr.gym_make('LunarLander-v3', 'LL3_Better', render_fps=30)
score = run_episode(better_agent_function, env_record, verbose=False)
env_record.close()

print(f"Điểm số của lượt quay video: {score:.2f}")
dr.show('LL3_Better')

/usr/local/lib/python3.13/dist-packages/gymnasium/wrappers/rendering.py:292: UserWarning: WARN: Overwriting existing videos at /content/videos folder (try specifying a different `video_folder` for the `RecordVideo` wrapper if this is not desired)
  logger.warn(


Điểm số của lượt quay video: 100.00


Chạy đánh giá hiệu năng của từng loại tác tử trên 100 episode.

In [227]:
# Chạy đánh giá cho random_agent_function trên 100 lượt
env_random = gym.make("LunarLander-v3", render_mode=None)
rewards_random = run_episodes(random_agent_function, env_random, n=100)
env_random.close()

# Chạy đánh giá cho rocket_agent_function trên 100 lượt
env_rocket = gym.make("LunarLander-v3", render_mode=None)
rewards_rocket = run_episodes(rocket_agent_function, env_rocket, n=100)
env_rocket.close()

# Chạy đánh giá cho better_agent_function trên 100 lượt
env_better = gym.make("LunarLander-v3", render_mode=None)
rewards_better = run_episodes(better_agent_function, env_better, n=100)
env_better.close()

# In kết quả thống kê tạm thời để kiểm tra
print(f"Average reward (Random Agent - 100 episodes): {np.average(rewards_random):.2f}")
print(f"Success rate (Random Agent - 100 episodes): {np.sum(np.array(rewards_random) >= 100)}/100 ({np.sum(np.array(rewards_random) >= 100)}%)\n")

print(f"Average reward (Rocket Agent - 100 episodes): {np.average(rewards_rocket):.2f}")
print(f"Success rate (Rocket Agent - 100 episodes): {np.sum(np.array(rewards_rocket) >= 100)}/100 ({np.sum(np.array(rewards_rocket) >= 100)}%)\n")

print(f"Average reward (Better Agent - 100 episodes): {np.average(rewards_better):.2f}")
print(f"Success rate (Better Agent - 100 episodes): {np.sum(np.array(rewards_better) >= 100)}/100 ({np.sum(np.array(rewards_better) >= 100)}%)")

Average reward (Random Agent - 100 episodes): -100.00
Success rate (Random Agent - 100 episodes): 0/100 (0%)

Average reward (Rocket Agent - 100 episodes): -100.00
Success rate (Rocket Agent - 100 episodes): 0/100 (0%)

Average reward (Better Agent - 100 episodes): 26.00
Success rate (Better Agent - 100 episodes): 63/100 (63%)


## Đánh giá hiệu năng và Báo cáo

Chúng ta đã đánh giá ba tác nhân khác nhau trên 100 lượt chơi trong môi trường LunarLander-v3. Dưới đây là kết quả chi tiết của từng tác nhân:

### 0. Tác nhân `random_agent_function` (Random Agent)

*   **Điểm thưởng trung bình:** `-100.00`
*   **Tỷ lệ hạ cánh thành công:** `0/100 (0%)`

**Nhận xét:** Tác nhân này thực hiện các hành động hoàn toàn ngẫu nhiên. Dự kiến, nó không bao giờ có thể hạ cánh thành công và điểm trung bình sẽ rất thấp (gần -100), cho thấy nó luôn đâm vào bề mặt hoặc bay ra khỏi màn hình. Điều này xác nhận rằng một chiến lược ngẫu nhiên không hiệu quả trong môi trường này.

### 1. Tác nhân `rocket_agent_function` (Simple Agent)

*   **Điểm thưởng trung bình:** `-100.00`
*   **Tỷ lệ hạ cánh thành công:** `0/100 (0%)`

**Nhận xét:** Tác nhân đơn giản này chỉ sử dụng động cơ chính khi tàu rơi quá nhanh mà không tính đến góc nghiêng hay vị trí ngang. Kết quả cho thấy nó hầu như luôn thất bại (điểm trung bình rất thấp), với tỷ lệ thành công rất thấp hoặc bằng 0, chứng tỏ chiến lược này không đủ để hạ cánh an toàn. Mặc dù có đôi chút tốt hơn tác nhân ngẫu nhiên, độ ổn định của nó rất kém.

### 2. Tác nhân `better_agent_function` (Better Reflex-Based Agent)

*   **Điểm thưởng trung bình:** `26.00`
*   **Tỷ lệ hạ cánh thành công:** `63/100 (63%)`

**Nhận xét:** Tác nhân này là một bước cải tiến lớn so với các tác nhân trước. Nó đã cố gắng cân bằng góc nghiêng và điều khiển vị trí ngang bằng động cơ phụ, đồng thời hãm tốc độ rơi bằng động cơ chính. Với tỷ lệ thành công đáng kể và điểm trung bình dương, tác nhân này cho thấy một chiến lược điều khiển dựa trên luật và phản xạ có thể hiệu quả. Tuy nhiên, vẫn còn nhiều lượt chơi thất bại, cho thấy có những giới hạn trong khả năng ổn định và điều hướng, đặc biệt là trong các tình huống phức tạp hoặc với nhiễu loạn.

## Giải thích: Hạn chế của Tác nhân Phản xạ dựa trên Luật cục bộ trong Môi trường Liên tục

Các tác nhân phản xạ (reflex-based agents) hoạt động dựa trên một tập hợp các luật `IF-THEN` xác định trước, trong đó hành động được chọn trực tiếp từ trạng thái (hoặc một phần của trạng thái) hiện tại. Mặc dù chúng có thể hiệu quả trong các môi trường đơn giản hoặc khi được thiết kế kỹ lưỡng cho các tác vụ cụ thể, chúng gặp phải những thách thức đáng kể trong môi trường liên tục như LunarLander-v3 vì một số lý do:

1.  **Tính liên tục của không gian trạng thái và hành động:**
    *   Trong môi trường liên tục, số lượng trạng thái tiềm năng là vô hạn (ví dụ: vị trí `x, y`, vận tốc `vx, vy`, góc `angle` đều là các giá trị thực). Việc tạo ra các luật bao phủ mọi trạng thái có thể là không khả thi.
    *   Các tác nhân phản xạ thường ánh xạ trạng thái sang các hành động rời rạc (ví dụ: bật động cơ chính, trái, phải, không làm gì). Việc chuyển đổi từ trạng thái liên tục sang hành động rời rạc có thể bỏ qua thông tin quan trọng hoặc không cho phép điều khiển tinh tế cần thiết.

2.  **Thiếu khả năng dự đoán và lập kế hoạch:**
    *   Tác nhân phản xạ chỉ phản ứng với trạng thái hiện tại (`local rules`). Nó không có khả năng dự đoán hậu quả của hành động mình trong tương lai hoặc lập kế hoạch cho một chuỗi hành động để đạt được mục tiêu dài hạn (như hạ cánh). Trong LunarLander, việc hạ cánh thành công đòi hỏi phải kiểm soát đồng thời nhiều yếu tố (vận tốc, góc, vị trí) trong một khoảng thời gian, chứ không chỉ là phản ứng tức thì.
    *   Việc thiếu mô hình về môi trường hoặc khả năng học hỏi từ kinh nghiệm làm cho tác nhân không thể thích nghi với các tình huống mới hoặc tối ưu hóa chiến lược của mình.

3.  **Khó khăn trong việc thiết kế luật:**
    *   Việc viết các luật `IF-THEN` đủ tinh vi để xử lý tất cả các trường hợp trong môi trường liên tục là cực kỳ khó khăn và tốn thời gian. Các luật phải được điều chỉnh cẩn thận để tránh các xung đột hoặc hành vi không mong muốn.
    *   Một thay đổi nhỏ trong các ngưỡng hoặc hệ số trong luật có thể dẫn đến sự thay đổi lớn trong hiệu suất của tác nhân. Quá trình điều chỉnh này (tuning) thường là thử và sai.

4.  **Tính 'tham lam' (Greedy Nature):**
    *   Vì tác nhân chỉ tập trung vào việc tối ưu hóa hành động tại thời điểm hiện tại, nó có thể đưa ra các quyết định cục bộ tối ưu nhưng lại dẫn đến kết quả kém tối ưu trên toàn cục. Ví dụ, việc giảm tốc độ rơi ngay lập tức có thể khiến nó trôi quá xa điểm hạ cánh lý tưởng.

**Kết luận:**

Mặc dù tác nhân `better_agent_function` đã thể hiện khả năng vượt trội so với tác nhân ngẫu nhiên và tác nhân tên lửa đơn giản, tỷ lệ thành công 63% vẫn cho thấy hạn chế. Điều này là do nó vẫn dựa trên một tập hợp các luật cứng nhắc, không thể học hỏi hoặc thích nghi một cách linh hoạt như các tác nhân học tăng cường (Reinforcement Learning - RL) có thể làm trong các môi trường liên tục và phức tạp hơn.